# Can we predict 30-day readmission?
Target: `readmit_30d`. Prediction is made at discharge, so all recorded columns (admission labs/BMI, length of stay, discharge medications) are available. `person_key` is an identifier and is dropped. Each person has several visits, so the test set and CV folds hold **whole persons**. There is no calendar column, so no temporal split is possible; `visit_number` is kept as a feature (known at prediction time).

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score, confusion_matrix
df = pd.read_csv('data.csv')
print(df.shape, 'persons:', df.person_key.nunique(), 'prevalence:', df.readmit_30d.mean().round(3))
print(df.groupby('person_key').size().describe()[['min','mean','max']])

## 1. Hold out a test set (whole persons) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.person_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.person_key) & set(test.person_key)
print(len(train), len(test), train.readmit_30d.mean().round(3), test.readmit_30d.mean().round(3))
FEATURES = ['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
X, y, g = train[FEATURES], train.readmit_30d, train.person_key
# EDA on training rows only
print(train.groupby('readmit_30d')[[f for f in FEATURES if f!='sex']].mean().round(2))

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: standardised numeric + one-hot sex/postcode, L2, C in {0.01, 0.1, 1}.
3. `hgb`: histogram gradient boosting, small grid over learning rate / depth, strong regularisation.

Selection metric: out-of-fold ROC AUC. Decision threshold chosen on out-of-fold predictions only (maximising F1).

In [ ]:
num = ['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
def make(name, **p):
    if name=='baseline': return DummyClassifier(strategy='prior')
    if name=='logreg':
        ct = ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),['sex','postcode_area'])])
        return make_pipeline(ct, LogisticRegression(C=p['C'], max_iter=2000))
    ct = ColumnTransformer([('n','passthrough',num+['postcode_area']),('c',OneHotEncoder(handle_unknown='ignore'),['sex'])])
    return make_pipeline(ct, HistGradientBoostingClassifier(learning_rate=p['lr'], max_depth=p['d'], max_leaf_nodes=8, l2_regularization=5.0, max_iter=150, random_state=0))
candidates = [('baseline',{})]+[('logreg',{'C':C}) for C in (0.01,0.1,1)] + \
  [('hgb',{'lr':a['lr'],'d':a['d']}) for a in ParameterGrid({'lr':[0.03,0.1],'d':[2,4]})]
print(len(candidates),'candidates')

## 3. Grouped, stratified CV inside the training rows (pipelines are refit per fold, so preprocessing is fit in-fold)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1)
folds = list(cv.split(X, y, g))
rows, oof_all = [], {}
for name, p in candidates:
    oof = np.zeros(len(train)); fa=[]
    for a,b in folds:
        m = make(name, **p).fit(X.iloc[a], y.iloc[a])
        oof[b] = m.predict_proba(X.iloc[b])[:,1]; fa.append(roc_auc_score(y.iloc[b], oof[b]))
    key=f'{name} {p}'; oof_all[key]=oof
    rows.append(dict(model=key, cv_auc=np.mean(fa), cv_auc_sd=np.std(fa), oof_ap=average_precision_score(y,oof), oof_brier=brier_score_loss(y,oof)))
res = pd.DataFrame(rows).sort_values('cv_auc', ascending=False); res.round(4)

## 4. Choose model and threshold on CV only

In [ ]:
best_key = res.iloc[0].model
best = next((n,p) for n,p in candidates if f'{n} {p}'==best_key)
oof = oof_all[best_key]
ths = np.linspace(0.05,0.6,56)
f1s = [f1_score(y, oof>=t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen:', best_key, '| threshold (max OOF F1):', round(THRESH,3), '| OOF F1:', round(max(f1s),3))
print('F1 of flag-everyone:', round(f1_score(y, np.ones(len(y))),3))
CHOICE = dict(model=best_key, threshold=THRESH, cv_auc=float(res.iloc[0].cv_auc))  # recorded before the test set is touched
print(CHOICE)

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
final = make(best[0], **best[1]).fit(X, y)
p = final.predict_proba(test[FEATURES])[:,1]; yt = test.readmit_30d
rng = np.random.default_rng(0); pid = test.person_key.values; ups = np.unique(pid)
idx = {u:np.where(pid==u)[0] for u in ups}
boots=[]
for _ in range(500):
    i = np.concatenate([idx[u] for u in rng.choice(ups,len(ups))])
    boots.append(roc_auc_score(yt.values[i], p[i]))
print('TEST AUC', round(roc_auc_score(yt,p),3), 'person-bootstrap 95% CI', np.percentile(boots,[2.5,97.5]).round(3))
print('TEST AP', round(average_precision_score(yt,p),3), '(prevalence', round(yt.mean(),3),')')
print('TEST Brier', round(brier_score_loss(yt,p),4), 'vs prevalence-only', round(brier_score_loss(yt,np.full(len(yt),y.mean())),4))
print('At threshold', round(THRESH,3), 'F1', round(f1_score(yt,p>=THRESH),3)); print(confusion_matrix(yt,p>=THRESH))